# 3주차 정답 Notebook — Pandas로 공정 데이터 읽고 선택하기

## 1단계. Pandas 불러오기

In [1]:
import pandas as pd
print('Pandas 준비 완료')

Pandas 준비 완료


## 2단계. 데이터 읽기

In [2]:
df = pd.read_csv("../../data/weekly/week03/week03_process_filtering.csv")
df.head()

,측정시간,로트번호,설비번호,공정명,온도_섭씨,압력_Pa,가스유량_slm,합격여부
0,2024-03-01 00:00,LOT-0217,EQ-01,산화,297.5,1002.7,51.20,1
1,2024-03-01 03:00,LOT-0230,EQ-03,산화,297.1,988.5,48.33,1
2,2024-03-01 06:00,LOT-0231,EQ-02,포토,295.9,1000.7,47.47,1
3,2024-03-01 12:00,LOT-0220,EQ-02,증착,296.3,1007.2,54.65,1
4,2024-03-01 16:00,LOT-0303,EQ-02,증착,301.5,999.1,47.28,1


## 3단계. 열 선택하기

In [3]:
df["온도_섭씨"]

df[["온도_섭씨", "압력_Pa"]]

,온도_섭씨,압력_Pa
0,297.5,1002.7
1,297.1,988.5
2,295.9,1000.7
3,296.3,1007.2
4,301.5,999.1
...,...,...
95,298.4,1003.0
96,301.5,1012.4
97,300.3,1011.3
98,302.3,995.3


**질문 답**: 열 하나만 선택하면 Series(값 목록 하나) 형태이고, 두 열을 선택하면
DataFrame(표) 형태다.

In [4]:
df.dtypes

측정시간         object
로트번호         object
설비번호         object
공정명          object
온도_섭씨       float64
압력_Pa       float64
가스유량_slm    float64
합격여부          int64
dtype: object

**질문 답**: 온도_섭씨의 dtype은 float64이다.

## 4단계. 조건 검색 — 특정 설비 찾기

In [5]:
eq02 = df[df["설비번호"] == "EQ-02"]
print(eq02.shape)

(32, 8)


**질문 답**: EQ-02 기록은 총 32건이다.

## 5단계. 불합격 행 찾기

In [6]:
fail_eq02 = eq02[eq02["합격여부"] == -1]
fail_eq02[["측정시간", "로트번호", "공정명", "합격여부"]]

,측정시간,로트번호,공정명,합격여부
13,2024-03-04 09:00,LOT-0203,산화,-1
37,2024-03-11 09:00,LOT-0128,식각,-1
52,2024-03-16 11:00,LOT-0228,산화,-1
59,2024-03-19 06:00,LOT-0032,식각,-1
60,2024-03-19 18:00,LOT-0237,산화,-1
83,2024-03-28 12:00,LOT-0365,세정,-1
90,2024-03-31 13:00,LOT-0219,세정,-1
97,2024-04-03 04:00,LOT-0376,증착,-1


**질문 답**: EQ-02 불합격은 8건이며, 비율은 약 25.0%다. (8 ÷ 32 × 100)

## 6단계. 온도 기준으로 정렬하기

In [7]:
df.sort_values("온도_섭씨", ascending=False).head(5)

,측정시간,로트번호,설비번호,공정명,온도_섭씨,압력_Pa,가스유량_slm,합격여부
15,2024-03-05 12:00,LOT-0041,EQ-04,산화,312.4,996.7,51.58,1
40,2024-03-11 22:00,LOT-0362,EQ-02,세정,309.1,1025.4,50.11,1
14,2024-03-05 03:00,LOT-0338,EQ-03,산화,308.9,1024.0,50.01,1
81,2024-03-27 12:00,LOT-0340,EQ-02,증착,308.7,1015.4,51.84,1
89,2024-03-31 10:00,LOT-0399,EQ-03,식각,306.8,1005.2,51.36,1


**질문 답**: 온도가 가장 높았던 상위 5건은 모두 합격(1)이었다. 이 결과만으로는
"온도가 높으면 항상 불합격이다"라고 말할 수 없다. 이번 데이터에서는 오히려 최고 온도
구간이 모두 합격 처리되었으므로, 온도와 불합격의 관계는 더 자세히 살펴봐야 한다
(8주차에서 상관관계와 원인 후보를 다룬다).

## 7단계(도전). 나머지 설비와 비교하기

In [8]:
others = df[df["설비번호"] != "EQ-02"]
fail_rate_eq02 = (eq02["합격여부"] == -1).mean() * 100
fail_rate_others = (others["합격여부"] == -1).mean() * 100
print(f"EQ-02 불합격 비율: {fail_rate_eq02:.1f}%")
print(f"나머지 설비 불합격 비율: {fail_rate_others:.1f}%")

EQ-02 불합격 비율: 25.0%
나머지 설비 불합격 비율: 5.9%


**결과**: EQ-02 불합격 비율 25.0%, 나머지 설비 불합격 비율 5.9%.

## 8단계. 오늘의 분석을 한 문장으로 정리하기

> EQ-02 설비의 공정 기록 32건 중 8건(25.0%)이 불합격했는데, 이는 나머지 설비의
> 평균 불합격 비율(5.9%)보다 훨씬 높은 수치다. EQ-02는 추가 점검이 필요해 보인다.